# 🧹 [EN] Phase 1: Descriptive Analytics (Data Preparation & EDA)
# 🧹 [ES] Fase 1: Analítica Descriptiva (Preparación de Datos y EDA)

> **[EN] Business Objective:** Clean the raw supermarket data and perform exploratory data analysis to understand historical performance and identify key trends.
> 
> **[ES] Objetivo de Negocio:** Limpiar los datos crudos del supermercado y realizar un análisis exploratorio para entender el rendimiento histórico e identificar tendencias clave.

---

In [2]:
import pandas as pd
import logging
from pathlib import Path
from typing import Optional

# 1. [EN] Logger Configuration (Executive level tracking) 
#    [ES] Configuración de logging (Seguimiento a nivel ejecutivo)
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

def clean_supermarket_data(input_path: str, output_path: str) -> Optional[pd.DataFrame]:
    """
    [EN] Loads, cleans, and standardizes the supermarket sales dataset.
    [ES] Carga, limpia y estandariza el conjunto de datos de ventas del supermercado.
    
    Args:
        input_path (str): [EN] Path to the raw CSV file / [ES] Ruta al archivo CSV original.
        output_path (str): [EN] Target path for the processed CSV / [ES] Ruta destino para el CSV procesado.
        
    Returns:
        pd.DataFrame: [EN] Cleaned dataframe or None if an error occurs.
                      [ES] Dataframe limpio o None si ocurre un error.
    """
    try:
        # 2. [EN] Path Validation / [ES] Validación de rutas
        raw_file = Path(input_path)
        if not raw_file.exists():
            logging.error(f"[EN] Source file not found / [ES] Archivo origen no encontrado: {input_path}")
            return None
            
        logging.info("[EN] Loading raw dataset... / [ES] Cargando dataset crudo...")
        
        # [EN] Using memory_map for performance optimization on larger datasets
        # [ES] Usando memory_map para optimizar el rendimiento en datasets más grandes
        df = pd.read_csv(raw_file, memory_map=True)
        
        # 3. [EN] Memory optimization: Work on a copy / [ES] Optimización: Trabajar en una copia aislada
        df_clean = df.copy()
        
        # 4. [EN] Column Standardization (snake_case) / [ES] Estandarización de columnas (snake_case)
        #    [EN] Using .strip() to remove trailing spaces / [ES] Usando .strip() para remover espacios extra
        df_clean.columns = [str(col).strip().lower().replace(' ', '_') for col in df_clean.columns]
        
        logging.info("[EN] Transforming data types (Dates & Times)... / [ES] Transformando tipos de datos...")
        
        # 5. [EN] Data type casting / [ES] Conversión y validación de tipos
        df_clean['date'] = pd.to_datetime(df_clean['date'])
        
        # [EN] 'mixed' format prevents warnings and infers PM/AM / [ES] El formato 'mixed' previene advertencias y deduce PM/AM
        df_clean['time'] = pd.to_datetime(df_clean['time'], format='mixed').dt.time
        
        # 6. [EN] Safe Export / [ES] Exportación segura
        out_file = Path(output_path)
        
        # [EN] Ensure target directory exists / [ES] Asegurar que el directorio destino exista
        out_file.parent.mkdir(parents=True, exist_ok=True) 
        
        df_clean.to_csv(out_file, index=False)
        logging.info(f"[EN] Pipeline completed. Data saved to / [ES] Pipeline completado. Datos en: {output_path}")
        
        return df_clean
        
    except Exception as e:
        # [EN] Catch unexpected errors gracefully / [ES] Capturar errores inesperados elegantemente
        logging.error(f"[EN] Execution failed / [ES] Fallo en la ejecución: {str(e)}")
        return None

# --- [EN] Pipeline Execution / [ES] Ejecución del Pipeline ---
if __name__ == "__main__":
    # [EN] Define project paths / [ES] Definir rutas del proyecto
    INPUT_DIR = "../data/raw/supermarket_analysis.csv"
    OUTPUT_DIR = "../data/processed/supermarket_analysis_clean.csv"
    
    # [EN] Run the ETL function / [ES] Ejecutar la función de extracción y transformación
    df_final = clean_supermarket_data(INPUT_DIR, OUTPUT_DIR)
    
    # [EN] Display top 5 rows if successful / [ES] Mostrar los 5 primeros registros si fue exitoso
    if df_final is not None:
        display(df_final.head())

2026-09-14 14:36:45 - INFO - [EN] Loading raw dataset... / [ES] Cargando dataset crudo...
2026-09-14 14:36:45 - INFO - [EN] Transforming data types (Dates & Times)... / [ES] Transformando tipos de datos...
2026-09-14 14:36:45 - INFO - [EN] Pipeline completed. Data saved to / [ES] Pipeline completado. Datos en: ../data/processed/supermarket_analysis_clean.csv


,invoice_id,branch,city,customer_type,gender,product_line,unit_price,quantity,tax_5%,sales,date,time,payment,cogs,gross_margin_percentage,gross_income,rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,2019-01-05,13:08:00,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,2019-03-08,10:29:00,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,2019-03-03,13:23:00,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,2019-01-27,20:33:00,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2019-02-08,10:37:00,Ewallet,604.17,4.761905,30.2085,5.3


### 📈 [EN] Descriptive Statistics / [ES] Estadística Descriptiva
> **[EN]** Before visual exploration, we analyze the central tendency, dispersion, and shape of the dataset's distribution to validate data integrity.
> **[ES]** Antes de la exploración visual, analizamos la tendencia central, dispersión y forma de la distribución del conjunto de datos para validar su integridad.

In [4]:
import pandas as pd
import numpy as np
import logging

def compute_advanced_statistics(df: pd.DataFrame):
    """
    [EN] Computes an advanced statistical summary including central tendency, dispersion, and distribution shape (skewness, kurtosis).
    [ES] Calcula un resumen estadístico avanzado incluyendo tendencia central, dispersión y forma de la distribución (asimetría, curtosis).
    """
    logging.info("[EN] Computing advanced statistical summary... / [ES] Computando resumen estadístico avanzado...")
    
    # --- 1. [EN] Numeric Variables / [ES] Variables Numéricas ---
    print("=" * 90)
    print("📊 [EN] ADVANCED NUMERICAL STATISTICS / [ES] ESTADÍSTICA NUMÉRICA AVANZADA")
    print("=" * 90)
    
    numeric_cols = df.select_dtypes(include=[np.number])
    if not numeric_cols.empty:
        # Calcular métricas base y transponer
        stats_df = numeric_cols.describe().T
        
        # Inyectar métricas de forma de distribución (Nivel PhD)
        stats_df['skewness'] = numeric_cols.skew()
        stats_df['kurtosis'] = numeric_cols.kurtosis()
        stats_df['missing_pct'] = (numeric_cols.isnull().sum() / len(df)) * 100
        
        # Reordenar columnas para una lectura lógica (Tendencia -> Dispersión -> Forma)
        cols_order = ['count', 'missing_pct', 'mean', '50%', 'std', 'min', 'max', 'skewness', 'kurtosis']
        stats_df = stats_df[[c for c in cols_order if c in stats_df.columns]]
        
        # Formateo visual avanzado (Doble Heatmap)
        styled_stats = (stats_df.style
                        .format("{:.3f}")
                        .background_gradient(subset=['mean', '50%', 'max'], cmap='YlGnBu') # Azul/Verde para magnitudes
                        .background_gradient(subset=['std', 'skewness', 'kurtosis'], cmap='OrRd')) # Rojo para alertas de dispersión/sesgo
        display(styled_stats)
    else:
        print("[EN] No numeric columns found. / [ES] No se encontraron columnas numéricas.")
        
    # --- 2. [EN] Categorical Variables / [ES] Variables Categóricas ---
    print("\n" + "=" * 90)
    print("🏷️ [EN] CATEGORICAL STATISTICS / [ES] ESTADÍSTICA CATEGÓRICA")
    print("=" * 90)
    
    categorical_cols = df.select_dtypes(include=['object', 'category'])
    if not categorical_cols.empty:
        cat_stats = categorical_cols.describe().T
        cat_stats['missing_pct'] = (categorical_cols.isnull().sum() / len(df)) * 100
        display(cat_stats)
    else:
        print("[EN] No categorical columns found. / [ES] No se encontraron columnas categóricas.")

# --- Ejecución ---
if 'df_final' in locals() and df_final is not None:
    compute_advanced_statistics(df_final)
else:
    logging.error("[EN] DataFrame 'df_final' is not defined. / [ES] El DataFrame 'df_final' no está definido.")

2026-09-14 14:36:45 - INFO - [EN] Computing advanced statistical summary... / [ES] Computando resumen estadístico avanzado...


📊 [EN] ADVANCED NUMERICAL STATISTICS / [ES] ESTADÍSTICA NUMÉRICA AVANZADA


,count,missing_pct,mean,50%,std,min,max,skewness,kurtosis
unit_price,1000.000,0.000,55.672,55.230,26.495,10.080,99.960,0.007,-1.219
quantity,1000.000,0.000,5.510,5.000,2.923,1.000,10.000,0.013,-1.216
tax_5%,1000.000,0.000,15.379,12.088,11.709,0.508,49.650,0.893,-0.082
sales,1000.000,0.000,322.967,253.848,245.885,10.678,1042.650,0.893,-0.082
cogs,1000.000,0.000,307.587,241.760,234.177,10.170,993.000,0.893,-0.082
gross_margin_percentage,1000.000,0.000,4.762,4.762,0.000,4.762,4.762,0.000,0.000
gross_income,1000.000,0.000,15.379,12.088,11.709,0.508,49.650,0.893,-0.082
rating,1000.000,0.000,6.973,7.000,1.719,4.000,10.000,0.009,-1.152



🏷️ [EN] CATEGORICAL STATISTICS / [ES] ESTADÍSTICA CATEGÓRICA


,count,unique,top,freq,missing_pct
invoice_id,1000,1000,849-09-3807,1,0.0
branch,1000,3,Alex,340,0.0
city,1000,3,Yangon,340,0.0
customer_type,1000,2,Member,565,0.0
gender,1000,2,Female,571,0.0
product_line,1000,6,Fashion accessories,178,0.0
time,1000,506,19:48:00,7,0.0
payment,1000,3,Ewallet,345,0.0


In [5]:
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import pandas as pd
import logging

# --- [EN] Plotly Configuration / [ES] Configuración de Plotly ---

# [EN] Fix for Jupyter JS rendering / [ES] Solución para el renderizado JS en Jupyter
pio.renderers.default = 'iframe' 

# [EN] Configure visual theme / [ES] Configurar el tema visual
pio.templates.default = "plotly_white"

def plot_sales_by_branch(df: pd.DataFrame) -> go.Figure:
    """
    [EN] Generates an interactive bar chart of total sales aggregated by branch and city.
    [ES] Genera un gráfico de barras interactivo de ingresos totales agrupados por sucursal y ciudad.
    
    Args:
        df (pd.DataFrame): [EN] Cleaned dataset / [ES] Conjunto de datos limpio.
        
    Returns:
        go.Figure: [EN] Plotly figure object / [ES] Objeto de figura de Plotly.
    """
    logging.info("[EN] Calculating total sales by branch... / [ES] Calculando ingresos totales por sucursal...")
    
    try:
        # [EN] Data aggregation: Sum 'sales' grouped by 'branch' and 'city'
        # [ES] Agregación de datos: Sumar 'sales' agrupado por 'branch' y 'city'
        branch_sales = df.groupby(['branch', 'city'])['sales'].sum().reset_index()
        
        # [EN] Sort values descending to highlight top performers
        # [ES] Ordenar valores de forma descendente para destacar los mejores resultados
        branch_sales = branch_sales.sort_values(by='sales', ascending=False)
        
        # [EN] Create interactive bar chart
        # [ES] Crear gráfico de barras interactivo
        fig = px.bar(
            branch_sales, 
            x='branch', 
            y='sales', 
            color='city',
            title='[EN] Total Revenue by Branch / [ES] Ingresos Totales por Sucursal',
            text_auto='.2s', 
            labels={
                'branch': '[EN] Branch / [ES] Sucursal', 
                'sales': '[EN] Revenue ($) / [ES] Ingresos ($)', 
                'city': '[EN] City / [ES] Ciudad'
            },
            color_discrete_sequence=px.colors.qualitative.Prism
        )
        
        # [EN] Fine-tune layout for executive presentation
        # [ES] Ajustes finos de diseño para presentación ejecutiva
        fig.update_layout(
            title_font_size=20,
            title_x=0.5, 
            xaxis_tickangle=0,
            showlegend=True,
            margin=dict(t=60, b=40, l=40, r=40)
        )
        
        logging.info("[EN] Chart generated successfully. / [ES] Gráfico generado exitosamente.")
        return fig
        
    except Exception as e:
        logging.error(f"[EN] Error generating chart / [ES] Error al generar el gráfico: {str(e)}")
        raise

# --- [EN] Execution Block / [ES] Bloque de Ejecución ---
if 'df_final' in locals() and isinstance(df_final, pd.DataFrame):
    try:
        fig_branch = plot_sales_by_branch(df_final)
        fig_branch.show()
    except Exception as exec_error:
        logging.error(f"[EN] Execution failed / [ES] Fallo en la ejecución: {str(exec_error)}")
else:
    logging.warning("[EN] The dataframe 'df_final' is invalid. Execute the ETL cell first. / [ES] El dataframe 'df_final' es inválido. Ejecuta la celda de ETL primero.")

2026-09-14 14:36:47 - INFO - [EN] Calculating total sales by branch... / [ES] Calculando ingresos totales por sucursal...
2026-09-14 14:36:47 - INFO - [EN] Chart generated successfully. / [ES] Gráfico generado exitosamente.


In [6]:
import plotly.express as px
import pandas as pd
import logging
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe' 

def plot_income_by_product(df: pd.DataFrame) -> go.Figure:
    """
    [EN] Generates a horizontal bar chart of total gross income by product line.
    [ES] Genera un gráfico de barras horizontales del ingreso bruto total por línea de producto.
    """
    logging.info("[EN] Analyzing gross income by product... / [ES] Analizando ingreso bruto por producto...")
    
    try:
        # [EN] Group by product line and calculate total gross income
        # [ES] Agrupar por línea de producto y calcular el ingreso bruto total
        income_df = df.groupby('product_line')['gross_income'].sum().reset_index()
        income_df = income_df.sort_values(by='gross_income', ascending=True)
        
        fig = px.bar(
            income_df,
            x='gross_income',
            y='product_line',
            orientation='h',
            title='[EN] Total Gross Income by Product Line / [ES] Ingreso Bruto Total por Producto',
            text_auto='.2f', # [EN] 2 decimal places / [ES] 2 decimales
            labels={
                'gross_income': '[EN] Gross Income ($) / [ES] Ingreso Bruto ($)',
                'product_line': '[EN] Product Line / [ES] Línea de Producto'
            },
            color='gross_income',
            color_continuous_scale=px.colors.sequential.Teal
        )
        
        fig.update_layout(title_font_size=20, title_x=0.5, margin=dict(t=60, b=40, l=40, r=40))
        return fig
    except Exception as e:
        logging.error(f"Error: {str(e)}")
        raise

# --- Ejecución ---
if 'df_final' in locals() and isinstance(df_final, pd.DataFrame):
    fig_income = plot_income_by_product(df_final)
    fig_income.show()

2026-09-14 14:36:47 - INFO - [EN] Analyzing gross income by product... / [ES] Analizando ingreso bruto por producto...


In [7]:
import plotly.express as px
import pandas as pd
import logging
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe' 

def plot_sales_by_customer_type(df: pd.DataFrame) -> go.Figure:
    """
    [EN] Generates a pie chart showing revenue distribution by customer type.
    [ES] Genera un gráfico de pastel mostrando la distribución de ingresos por tipo de cliente.
    """
    logging.info("[EN] Analyzing revenue by customer type... / [ES] Analizando ingresos por tipo de cliente...")
    
    try:
        customer_sales = df.groupby('customer_type')['sales'].sum().reset_index()
        
        fig = px.pie(
            customer_sales,
            names='customer_type',
            values='sales',
            title='[EN] Revenue Distribution by Customer Type / [ES] Distribución de Ingresos por Tipo de Cliente',
            hole=0.4, 
            color_discrete_sequence=px.colors.qualitative.Pastel
        )
        
        fig.update_traces(textposition='inside', textinfo='percent+label')
        fig.update_layout(title_font_size=20, title_x=0.5)
        return fig
    except Exception as e:
        logging.error(f"Error: {str(e)}")
        raise

# --- Ejecución ---
if 'df_final' in locals() and isinstance(df_final, pd.DataFrame):
    fig_customer = plot_sales_by_customer_type(df_final)
    fig_customer.show()

2026-09-14 14:36:47 - INFO - [EN] Analyzing revenue by customer type... / [ES] Analizando ingresos por tipo de cliente...


---

## 👤 Applied Data Scientist & Researcher / Científico de Datos Aplicado e Investigador

**Pablo Alberto Santana Flores**  
*Chemical Engineer | PhD in Marine Sciences | Decision Intelligence*  

> **🇬🇧 EN:** Thank you for exploring this phase of the **Dynamic Sales Prediction Engine**. This notebook showcases the *Descriptive* and *Predictive* foundations of our end-to-end analytical pipeline (ETL, EDA, and Meta Prophet forecasting). Building upon these insights, the project culminates in **Phase 3: Prescriptive Analytics**, where we utilize linear programming (`PuLP`) to translate sales forecasts into optimal inventory purchasing decisions to maximize profit. Feel free to connect on LinkedIn to discuss the technical architecture.  
> 
> **🇲🇽 ES:** Gracias por explorar esta fase del **Motor Dinámico de Predicción de Ventas**. Este notebook exhibe las bases *Descriptivas* y *Predictivas* de nuestro pipeline analítico de extremo a extremo (ETL, EDA y pronóstico con Meta Prophet). Partiendo de estos *insights*, el proyecto culmina en la **Fase 3: Analítica Prescriptiva**, donde utilizamos programación lineal (`PuLP`) para traducir los pronósticos de ventas en decisiones óptimas de compra de inventario para maximizar ganancias. Siéntete libre de conectar en LinkedIn para discutir la arquitectura técnica.

*   **LinkedIn:** [linkedin.com/in/pablo-santana-mx](https://mx.linkedin.com/in/pablo-santana-mx)
*   **GitHub:** [github.com/Pablo-Santana-MX](https://github.com/Pablo-Santana-MX)